# Accelerating Donut: a benchmark-backed story

This notebook tells the story of taking the **Donut** document-understanding
model and progressively accelerating it, with every claim backed by an
experiment run on an H100. It reads the JSON logged by the sweep scripts in
`scripts/exp/` — run those first:

```bash
bash scripts/exp/exp_scaling.sh        # image x batch x preset grid (+ peak memory)
bash scripts/exp/exp_token_scaling.sh  # decode-length sweep
bash scripts/exp/exp_kernels.sh        # isolated attention-kernel microbench
```

The narrative arc:

1. **Baseline** Donut — where do we start?
2. **Mask cache** (the `eager` preset) — the first, free acceleration.
3. **`sdpa` and `fa`** — what they are, how they hit the encoder vs the decoder.
4. **Scaling** — image size, batch size, decode length; latency *and* memory.
5. **The flash-attention investigation** — why `fa` disappoints on this model,
   established rigorously (peak memory, which backend `sdpa` really uses, and
   the exact kernel-level reason).

In [ ]:
import json
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import seaborn as sns

sns.set_theme(style="darkgrid")

RESULTS = Path("../results")

# Canonical preset order -> stable colors across every chart in this notebook.
PRESETS = [
    "baseline",
    "eager",
    "sdpa",
    "sdpa_flash",
    "sdpa_math",
    "sdpa_efficient",
    "sdpa_cudnn",
    "fa",
]
PALETTE = dict(zip(PRESETS, sns.color_palette("tab10", n_colors=len(PRESETS))))


def load_speed(name: str) -> pd.DataFrame:
    # Glob the per-config JSON written by bench_speed.py into one dataframe.
    files = sorted((RESULTS / name).glob("*.json"))
    if not files:
        print(
            f"NO DATA in results/{name}/ -- run scripts/exp/{name}.sh on the GPU box first"
        )
        return pd.DataFrame()
    df = pd.json_normalize([json.loads(f.read_text()) for f in files])
    return df


def load_kernels() -> pd.DataFrame:
    # Flatten exp_kernels JSON to one row per (mode, kv_len, bs, kernel).
    f = RESULTS / "exp_kernels" / "bench_attention_kernels.json"
    if not f.exists():
        print("NO DATA -- run scripts/exp/exp_kernels.sh first")
        return pd.DataFrame()
    raw = json.loads(f.read_text())
    rows = []
    for rec in raw["records"]:
        for kernel, v in rec["kernels"].items():
            rows.append(
                {
                    "mode": rec["mode"],
                    "kv_len": rec["kv_len"],
                    "batch_size": rec["batch_size"],
                    "kernel": kernel,
                    "status": v["status"],
                    "mean_ms": v.get("mean_ms"),
                    "std_ms": v.get("std_ms"),
                }
            )
    return pd.DataFrame(rows), raw["meta"]


def order(values):
    # Sort an iterable of preset names into canonical order.
    return [p for p in PRESETS if p in set(values)]


def propagate_std(df, mean_col, std_col, rate_col):
    # Delta-method std for a reciprocal-style rate metric (rate ~ C / mean_col):
    # relative error is preserved, so std_rate ~= rate * (std_mean / mean).
    return df[rate_col] * (df[std_col] / df[mean_col])


def lineplot_by(
    ax, df, x, y, hue, *, palette=None, log_x=False, log_y=False, xticks=None, yerr=None
):
    # sns.lineplot with markers per hue level, exact ticks at the real data
    # values (not wherever matplotlib's log-scale default lands).
    sns.lineplot(
        ax=ax,
        data=df,
        x=x,
        y=y,
        hue=hue,
        style=hue,
        hue_order=order(df[hue].unique()) if palette else None,
        palette=palette,
        markers=True,
        dashes=False,
    )
    if yerr is not None:
        fallback = sns.color_palette(n_colors=df[hue].nunique())
        for j, (level, g) in enumerate(df.groupby(hue, sort=False)):
            g = g.sort_values(x)
            color = (palette or {}).get(level, fallback[j % len(fallback)])
            ax.fill_between(
                g[x],
                g[y] - g[yerr],
                g[y] + g[yerr],
                color=color,
                alpha=0.15,
                linewidth=0,
            )
    if log_x:
        ax.set_xscale("log")
    if log_y:
        ax.set_yscale("log")
    # Explicit ticks at the real swept values only make sense for a numeric
    # axis -- a categorical/string x (e.g. "32x32" size labels) already gets
    # correct positions and labels from seaborn, overriding them breaks it.
    if pd.api.types.is_numeric_dtype(df[x]):
        ticks = xticks if xticks is not None else sorted(df[x].unique())
        ax.set_xticks(ticks)
        ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
        ax.xaxis.set_minor_formatter(mticker.NullFormatter())


scaling = load_speed("exp_scaling")
tokens = load_speed("exp_token_scaling")
scaling.head() if not scaling.empty else None

## 0. The model

Donut (`naver-clova-ix/donut-base`) is an OCR-free document model: pixels in,
text out, no separate text-detection stage. Two halves:

- **Encoder — DonutSwin** (a Swin vision transformer). Attention is *windowed*:
  each token only attends within a fixed `window_size² = 100`-token window, plus
  a relative-position bias and a cyclic-shift mask. Image resolution changes the
  *number* of windows, not the per-window sequence length.
- **Decoder — MBart** (a standard autoregressive text decoder). With KV caching
  on (always, here), it emits one token at a time: every decode step is a
  `query_len = 1` attention call against a growing key/value cache.

That asymmetry — windowed-100-token encoder vs `query_len=1` decoder — is the
thread that explains every result below.

## 1. Baseline: where we start

`baseline` applies **no** acceleration at all — plain eager PyTorch attention in
both halves, the Swin window mask recomputed on every forward. This is the
reference every speedup is measured against.

In [ ]:
def baseline_snapshot(df):
    if df.empty:
        return df
    b = df[(df.backend == "baseline") & (df.status == "ok")]
    cols = [
        "image_height",
        "image_width",
        "batch_size",
        "max_new_tokens",
        "encoder.mean_ms",
        "encoder.images_per_s",
        "encoder.peak_mem_mb",
        "generate.mean_ms",
        "generate.tokens_per_s",
        "generate.peak_mem_mb",
    ]
    return (
        b[[c for c in cols if c in b.columns]]
        .sort_values(["image_height", "batch_size"])
        .reset_index(drop=True)
    )


baseline_snapshot(scaling)

## 2. The first acceleration: caching the Swin window mask (`eager`)

The `eager` preset changes exactly one thing: DonutSwin recomputes its
cyclic-shift window mask on *every* forward pass, even though it only depends on
`(height, width, dtype)` and never changes. `mask_cache.py` computes it once and
reuses it. Nothing about the math changes — it's pure wasted-work removal — so
it's the cleanest "free win" in the stack.

Below: `baseline` vs `eager` for the encoder (where the mask lives).

In [ ]:
def baseline_vs_eager(df):
    if df.empty:
        return
    sub = df[
        (df.backend.isin(["baseline", "eager"]))
        & (df.status == "ok")
        & (df.batch_size == df.batch_size.min())
    ]
    if sub.empty:
        print("need baseline + eager rows")
        return
    sub = sub.assign(
        size_label=sub.image_height.astype(str) + "x" + sub.image_width.astype(str)
    )

    piv = sub.pivot_table(
        index="size_label", columns="backend", values="encoder.mean_ms"
    )
    piv = piv.reindex(
        sub.sort_values(["image_height", "image_width"]).size_label.unique()
    )
    piv["speedup"] = piv["baseline"] / piv["eager"]
    display(piv.round(3))

    fig, ax = plt.subplots(figsize=(7, 4))
    lineplot_by(
        ax,
        sub,
        x="size_label",
        y="encoder.mean_ms",
        hue="backend",
        palette=PALETTE,
        yerr="encoder.std_ms",
    )
    ax.set_ylabel("encoder mean ms")
    ax.set_xlabel("image size")
    ax.set_title("Mask cache: encoder latency, baseline vs eager")
    plt.show()


baseline_vs_eager(scaling)

## 3. `sdpa` and `fa`: what they are and where they apply

Both replace the eager attention with a fused kernel, but they hit different
halves of the model:

| preset | encoder | decoder |
|---|---|---|
| `sdpa` | `F.scaled_dot_product_attention`, PyTorch auto-picks the backend | `F.scaled_dot_product_attention`, auto-picked backend |
| `fa` | **same SDPA patch as `sdpa`** (DonutSwin has no flash path) | the real **FlashAttention-4** CUTLASS kernel (`flash_attn.cute`) |

The crucial detail: **`fa` differs from `sdpa` only in the decoder kernel.** The
encoder is patched identically in both. So any `fa` vs `sdpa` difference is a
clean, decoder-only comparison. (See `docs/attention-backends.md` for the full
mechanism.)

`sdpa`'s dispatcher can pick one of four backends — **math**, **efficient**,
**flash**, **cudnn** — based on shape/dtype/mask. The `sdpa_flash` / `sdpa_math`
/ `sdpa_efficient` / `sdpa_cudnn` presets *force* one each, which lets us later
prove which one plain `sdpa` actually lands on.

### SDPA benchmarking capability: isolating the kernels

`scripts/bench_attention_kernels.py` strips the model away entirely and races
the raw FA4 kernel against each forced SDPA backend on synthetic q/k/v at the
*real* decoder shape (16 heads, head_dim 64, bf16). Two regimes:

- **decode** — `query_len = 1` (what `generate()` actually does every step).
- **prefill** — `query_len = kv_len`, causal (the regime flash is designed for).

In [ ]:
def plot_kernels(bs=1):
    out = load_kernels()
    if isinstance(out, pd.DataFrame):  # empty sentinel
        return
    df, meta = out
    df = df[(df.batch_size == bs) & (df.status == "ok")]
    if df.empty:
        print("no ok kernel rows at this batch size")
        return
    modes = sorted(df["mode"].unique())
    fig, axes = plt.subplots(
        1, len(modes), figsize=(6.5 * len(modes), 4.5), squeeze=False
    )
    kv_lens = sorted(df.kv_len.unique())
    for ax, mode in zip(axes[0], modes):
        m = df[df["mode"] == mode]
        lineplot_by(
            ax,
            m,
            x="kv_len",
            y="mean_ms",
            hue="kernel",
            log_x=True,
            log_y=True,
            xticks=kv_lens,
            yerr="std_ms",
        )
        ax.set_xlabel("kv_len")
        ax.set_ylabel("mean ms")
        ax.set_title(
            f"{mode} (bs={bs}, heads={meta['num_heads']}, hd={meta['head_dim']})"
        )
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


plot_kernels(bs=1)

**Read the decode panel carefully** — this is the whole story in one chart. At
`query_len=1` (every real decode step), the FA4 kernel is *not* the fastest; the
cudnn / efficient / flash SDPA backends beat it, especially at small `kv_len`.
FA4 only pulls ahead in the **prefill** panel, and only once `kv_len` is large
(thousands). Hold that thought for section 5.

## 4. Scaling: image size, batch size, decode length

All eight presets, measured end-to-end on the real model.

In [ ]:
def plot_scaling_image(df, bs=1):
    # Throughput vs image size, one line per preset, fixed batch size.
    if df.empty:
        return
    sub = df[(df.status == "ok") & (df.batch_size == bs)]
    if sub.empty:
        print(f"no ok rows at bs={bs}")
        return
    sub = sub.assign(pixels=sub.image_height * sub.image_width)
    sub = sub.assign(
        size_label=sub.image_height.astype(str) + "x" + sub.image_width.astype(str)
    )
    sub = sub.assign(
        **{
            "encoder.images_per_s_std": propagate_std(
                sub, "encoder.mean_ms", "encoder.std_ms", "encoder.images_per_s"
            ),
            "generate.tokens_per_s_std": propagate_std(
                sub, "generate.mean_ms", "generate.std_ms", "generate.tokens_per_s"
            ),
        }
    )
    pixel_ticks = sorted(sub.pixels.unique())
    labels = (
        sub.drop_duplicates("pixels").set_index("pixels").loc[pixel_ticks, "size_label"]
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    lineplot_by(
        axes[0],
        sub,
        x="pixels",
        y="encoder.images_per_s",
        hue="backend",
        palette=PALETTE,
        xticks=pixel_ticks,
        yerr="encoder.images_per_s_std",
    )
    lineplot_by(
        axes[1],
        sub,
        x="pixels",
        y="generate.tokens_per_s",
        hue="backend",
        palette=PALETTE,
        xticks=pixel_ticks,
        yerr="generate.tokens_per_s_std",
    )
    for ax in axes:
        ax.set_xticklabels(labels, rotation=15)
        ax.set_xlabel("image size")
    axes[0].set_ylabel("encoder images/s")
    axes[1].set_ylabel("generate tokens/s")
    axes[0].set_title(f"Encoder throughput vs image size (bs={bs})")
    axes[1].set_title(f"Generation throughput vs image size (bs={bs})")
    axes[0].legend(fontsize=8, ncol=2)
    axes[1].legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()


plot_scaling_image(scaling, bs=1)

In [ ]:
def plot_scaling_batch(df):
    # Throughput + peak memory vs batch size at the native resolution.
    # (No error band on the memory panel: peak_mem_mb is a single reading
    # per config, not averaged over n_runs -- there's no variance to show.)
    if df.empty:
        return
    sub = df[df.status == "ok"]
    if sub.empty:
        return
    native = sub.assign(px=sub.image_height * sub.image_width).px.max()
    sub = sub[sub.image_height * sub.image_width == native]
    sub = sub.assign(
        **{
            "generate.tokens_per_s_std": propagate_std(
                sub, "generate.mean_ms", "generate.std_ms", "generate.tokens_per_s"
            )
        }
    )
    batch_ticks = sorted(sub.batch_size.unique())

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    lineplot_by(
        axes[0],
        sub,
        x="batch_size",
        y="generate.tokens_per_s",
        hue="backend",
        palette=PALETTE,
        xticks=batch_ticks,
        yerr="generate.tokens_per_s_std",
    )
    lineplot_by(
        axes[1],
        sub,
        x="batch_size",
        y="generate.peak_mem_mb",
        hue="backend",
        palette=PALETTE,
        xticks=batch_ticks,
    )
    axes[0].set_ylabel("generate tokens/s")
    axes[1].set_ylabel("generate peak mem (MB)")
    for ax in axes:
        ax.set_xlabel("batch size")
    axes[0].set_title("Generation throughput vs batch (native res)")
    axes[1].set_title("Generation peak memory vs batch (native res)")
    axes[0].legend(fontsize=8, ncol=2)
    axes[1].legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()


plot_scaling_batch(scaling)

In [ ]:
def plot_token_scaling(df):
    # Throughput and fa/sdpa gap vs decode length.
    # (No error band on the ratio panel: propagating error through a ratio
    # of two noisy means isn't worth the complexity for a secondary panel.)
    if df.empty:
        return
    sub = df[df.status == "ok"]
    if sub.empty:
        return
    sub = sub.assign(
        **{
            "generate.tokens_per_s_std": propagate_std(
                sub, "generate.mean_ms", "generate.std_ms", "generate.tokens_per_s"
            )
        }
    )
    mnt_ticks = sorted(sub.max_new_tokens.unique())

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    lineplot_by(
        axes[0],
        sub,
        x="max_new_tokens",
        y="generate.tokens_per_s",
        hue="backend",
        palette=PALETTE,
        log_x=True,
        xticks=mnt_ticks,
        yerr="generate.tokens_per_s_std",
    )
    axes[0].set_xlabel("max_new_tokens")
    axes[0].set_ylabel("generate tokens/s")
    axes[0].set_title("Throughput vs decode length")
    axes[0].legend(fontsize=8, ncol=2)

    # fa vs sdpa ratio: does the gap widen with more tokens?
    piv = sub.pivot_table(
        index="max_new_tokens", columns="backend", values="generate.tokens_per_s"
    )
    if {"fa", "sdpa"}.issubset(piv.columns):
        ratio = piv["sdpa"] / piv["fa"]
        sns.lineplot(
            ax=axes[1], x=ratio.index, y=ratio.values, marker="o", color="crimson"
        )
        axes[1].axhline(1.0, ls="--", color="gray")
        axes[1].set_xscale("log")
        axes[1].set_xticks(mnt_ticks)
        axes[1].xaxis.set_major_formatter(mticker.ScalarFormatter())
        axes[1].xaxis.set_minor_formatter(mticker.NullFormatter())
        axes[1].set_xlabel("max_new_tokens")
        axes[1].set_ylabel("sdpa / fa  tokens-per-s ratio")
        axes[1].set_title("How far fa trails sdpa, vs decode length")
    plt.tight_layout()
    plt.show()


plot_token_scaling(tokens)

## 5. The flash-attention investigation: why `fa` disappoints

`fa` is *supposed* to be the fastest preset. On this model it isn't. This
section establishes why, rigorously — three independent lines of evidence.

### 5a. Which backend does plain `sdpa` actually use?

We never have to guess. The forced-backend presets bracket it: whichever of
`sdpa_flash` / `sdpa_math` / `sdpa_efficient` / `sdpa_cudnn` matches plain
`sdpa`'s latency *is* what the auto-dispatcher picked.

In [ ]:
def autopick_table(df):
    if df.empty:
        return df
    sub = df[(df.status == "ok") & (df.batch_size == df.batch_size.min())]
    native = sub.assign(px=sub.image_height * sub.image_width).px.max()
    sub = sub[sub.image_height * sub.image_width == native]
    keep = ["sdpa", "sdpa_flash", "sdpa_math", "sdpa_efficient", "sdpa_cudnn", "fa"]
    sub = sub[sub.backend.isin(keep)]
    t = sub.set_index("backend")[
        ["generate.mean_ms", "generate.tokens_per_s", "generate.peak_mem_mb"]
    ]
    return t.reindex([b for b in keep if b in t.index]).round(3)


autopick_table(scaling)

Compare plain `sdpa`'s `generate.mean_ms` to the four forced backends: the one
it matches is the kernel it's dispatching to. (From earlier H100 runs this was
the native flash / cudnn path, *not* FA4.)

### 5b. Peak memory — is `fa` at least saving memory?

Flash attention's headline benefit is memory, not always speed. The
`generate.peak_mem_mb` column above (and the batch-memory plot in section 4)
is the fair test: if `fa` doesn't win on latency *and* doesn't save memory at
this shape, it has no advantage here at all.

### 5c. The root cause — it's the kernel, not overhead

Section 3's decode panel is the smoking gun: at `query_len=1, head_dim=64`, the
FA4 CUTLASS kernel is simply a *slower kernel* than cudnn/efficient — it loses
the isolated, model-free race. Combined with 5a (auto-`sdpa` picks one of those
faster kernels) the conclusion is airtight: `fa` loses because it forces a
kernel that's worse for this tiny decode shape, **not** because of any
HuggingFace / Python overhead.

And the regime where FA4 *would* win (long prefill, `kv_len` in the thousands —
section 3's prefill panel) is **unreachable** on `donut-base`: its decoder caps
at `max_position_embeddings = 1536`, so no real generation ever gets there. The
token-scaling plot (section 4) shows the consequence directly — the `sdpa / fa`
gap *widens* with more tokens; it never crosses over.

In [ ]:
def best_backend_heatmap(df):
    # For every (image size, batch size), which forced backend is fastest?
    # If the answer changes by shape, that's the visual reason auto-dispatch
    # (plain sdpa) beats every single static choice overall.
    if df.empty:
        return
    forced = ["sdpa_flash", "sdpa_math", "sdpa_efficient", "sdpa_cudnn"]
    sub = df[(df.status == "ok") & (df.backend.isin(forced))].copy()
    if sub.empty:
        print("need forced-backend rows")
        return
    sub["size_label"] = sub.image_height.astype(str) + "x" + sub.image_width.astype(str)
    winners = (
        sub.sort_values("generate.mean_ms")
        .groupby(["size_label", "batch_size"], as_index=False)
        .first()[["size_label", "batch_size", "backend"]]
    )
    grid = winners.pivot(index="size_label", columns="batch_size", values="backend")
    size_order = sub.sort_values(["image_height", "image_width"]).size_label.unique()
    grid = grid.reindex(size_order)

    codes = grid.apply(lambda col: col.map({b: i for i, b in enumerate(forced)}))
    cmap = mcolors.ListedColormap([PALETTE[b] for b in forced])
    fig, ax = plt.subplots(
        figsize=(1.6 * len(grid.columns) + 2, 0.9 * len(grid.index) + 1.5)
    )
    sns.heatmap(
        codes,
        annot=grid.values,
        fmt="",
        cmap=cmap,
        vmin=-0.5,
        vmax=len(forced) - 0.5,
        cbar=False,
        linewidths=1,
        linecolor="white",
        ax=ax,
    )
    ax.set_xlabel("batch size")
    ax.set_ylabel("image size")
    ax.set_title("Fastest forced SDPA backend, by shape")
    plt.tight_layout()
    plt.show()


best_backend_heatmap(scaling)

## 6. Correctness / faithfulness

Speed claims only matter if the accelerated presets still compute the *same
thing*. `scripts/audit_accel.py` checks exactly that: for every preset, it
diffs the encoder output and the full `generate()` output against a one-time
unaccelerated (`baseline`) reference, then reverts — `baseline` itself is
included as a built-in sanity check (it must diff to exactly zero against
itself).

In [ ]:
def load_audit():
    f = RESULTS / "audit_accel" / "audit_accel.json"
    if not f.exists():
        print("NO DATA -- run `uv run python scripts/audit_accel.py` first")
        return pd.DataFrame()
    raw = json.loads(f.read_text())
    rows = []
    for r in raw["records"]:
        if r["status"] != "ok":
            rows.append({"preset": r["preset"], "status": "error"})
            continue
        rows.append(
            {
                "preset": r["preset"],
                "status": "ok",
                "encoder_max_ae": r["encoder"]["max_ae"],
                "encoder_cosine_sim": r["encoder"]["cosine_sim"],
                "sequences_match": r["sequences_match"],
                "logits_max_ae": r["generate_logits"]["max_ae"],
            }
        )
    df = pd.DataFrame(rows).set_index("preset")
    return df.reindex([p for p in PRESETS if p in df.index])


load_audit()

In [ ]:
def scoreboard(df):
    # The "and the answer is..." chart: median generate() speedup vs
    # baseline, per preset, aggregated across the whole image x batch grid.
    if df.empty:
        return
    sub = df[df.status == "ok"]
    if sub.empty:
        return
    base = sub[sub.backend == "baseline"].set_index(
        ["image_height", "image_width", "batch_size"]
    )
    base_tps = base["generate.tokens_per_s"]
    rows = []
    for backend, g in sub.groupby("backend"):
        g = g.set_index(["image_height", "image_width", "batch_size"])
        common = g.index.intersection(base_tps.index)
        if len(common) == 0:
            continue
        speedup = (
            g.loc[common, "generate.tokens_per_s"] / base_tps.loc[common]
        ).median()
        rows.append({"backend": backend, "speedup": speedup})
    if not rows:
        print("need baseline + at least one other preset")
        return
    board = pd.DataFrame(rows).sort_values("speedup", ascending=True)

    fig, ax = plt.subplots(figsize=(7, 0.6 * len(board) + 1))
    colors = [PALETTE.get(b, "gray") for b in board.backend]
    ax.barh(board.backend, board.speedup, color=colors)
    ax.axvline(1.0, ls="--", color="gray")
    ax.set_xlabel("median generate() speedup vs baseline (all shapes)")
    ax.set_title("Scoreboard: which preset actually wins")
    plt.tight_layout()
    plt.show()


scoreboard(scaling)

## Conclusion

- **Mask cache (`eager`)** — a genuine free win on the encoder; no reason not to.
- **`sdpa`** — the real workhorse: fused attention in both halves, fastest
  overall, simplest config. Let PyTorch's dispatcher pick the backend.
- **`fa`** — the expected win that *isn't*, here. Donut's decoder is decode-bound
  (`query_len=1`), which is FlashAttention's worst case, and the 1536-position
  ceiling makes its favorable long-prefill regime unreachable. Forcing FA4 buys
  nothing — and can lose — on this model.

**Practical takeaway: ship plain `sdpa`.** Every reachable regime on this model
favors it.